In [5]:
import urllib.request
import os
from datetime import datetime

def download_vhi_data(save_dir="vhi_data"):
    os.makedirs(save_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    for province_id in range(1, 26):
        # Перевірка: чи вже є файл для цієї області
        existing = glob.glob(os.path.join(save_dir, f"vhi_province_{province_id}_*.csv"))
        if existing:
            print(f"[SKIP] Область {province_id} вже є: {existing[0]}")
            continue
        
        url = (
            f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?"
            f"country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        )
        
        filename = os.path.join(save_dir, f"vhi_province_{province_id}_{timestamp}.csv")
        
        try:
            urllib.request.urlretrieve(url, filename)
            print(f"[OK] Область {province_id} -> {filename}")
        except Exception as e:
            print(f"[ERROR] Область {province_id}: {e}")

download_vhi_data()

[SKIP] Область 1 вже є: vhi_data\vhi_province_1_20260422_145934.csv
[SKIP] Область 2 вже є: vhi_data\vhi_province_2_20260422_145940.csv
[SKIP] Область 3 вже є: vhi_data\vhi_province_3_20260422_145942.csv
[SKIP] Область 4 вже є: vhi_data\vhi_province_4_20260422_145944.csv
[SKIP] Область 5 вже є: vhi_data\vhi_province_5_20260422_145946.csv
[SKIP] Область 6 вже є: vhi_data\vhi_province_6_20260422_145948.csv
[SKIP] Область 7 вже є: vhi_data\vhi_province_7_20260422_145950.csv
[SKIP] Область 8 вже є: vhi_data\vhi_province_8_20260422_145951.csv
[SKIP] Область 9 вже є: vhi_data\vhi_province_9_20260422_145953.csv
[SKIP] Область 10 вже є: vhi_data\vhi_province_10_20260422_145954.csv
[SKIP] Область 11 вже є: vhi_data\vhi_province_11_20260422_145956.csv
[SKIP] Область 12 вже є: vhi_data\vhi_province_12_20260422_145958.csv
[SKIP] Область 13 вже є: vhi_data\vhi_province_13_20260422_145959.csv
[SKIP] Область 14 вже є: vhi_data\vhi_province_14_20260422_150001.csv
[SKIP] Область 15 вже є: vhi_data\vhi_

In [7]:
f = glob.glob("vhi_data/*.csv")[0]

with open(f, 'r', encoding='utf-8') as file:
    for i, line in enumerate(file):
        print(f"{i}: {line}", end='')
        if i > 10:
            break

0: Mean data for UKR  Province= 10: Khmel'nyts'kyy,  from 1981 to 2024, weekly; version='GC_current'<br>for cropland area only<br>
1: year,week, SMN,SMT,VCI,TCI, VHI<br>
2: <tt><pre>1982, 1, 0.059,258.24, 51.11, 48.78, 49.95,
3: 1982, 2, 0.063,261.53, 55.89, 38.20, 47.04,
4: 1982, 3, 0.063,263.45, 57.30, 32.69, 44.99,
5: 1982, 4, 0.061,265.10, 53.96, 28.62, 41.29,
6: 1982, 5, 0.058,266.42, 46.87, 28.57, 37.72,
7: 1982, 6, 0.056,267.47, 39.55, 30.27, 34.91,
8: 1982, 7, 0.055,268.58, 35.19, 31.10, 33.14,
9: 1982, 8, 0.057,270.15, 33.35, 32.09, 32.72,
10: 1982, 9, 0.057,271.60, 30.82, 34.71, 32.77,
11: 1982,10, 0.057,273.10, 27.66, 36.79, 32.23,


In [9]:
def clean_data(save_dir="vhi_data"):
    all_files = glob.glob(os.path.join(save_dir, "*.csv"))

    if not all_files:
        raise FileNotFoundError(f"Файли не знайдено в папці '{save_dir}'")

    list_of_dfs = []

    for filename in all_files:
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                lines = f.readlines()

            # Рядок 1 — заголовок (year,week, SMN,...)
            # Рядок 2+ — дані, але рядок 2 містить <tt><pre>
            # Очищуємо кожен рядок від HTML-тегів
            clean_lines = []
            for line in lines[1:]:  # пропускаємо рядок 0 (мета-інфо)
                line = line.replace('<tt><pre>', '').replace('<br>', '').strip()
                if line:
                    clean_lines.append(line)

            # Перший clean_lines[0] — це заголовок "year,week, SMN,..."
            from io import StringIO
            csv_text = "\n".join(clean_lines)
            df = pd.read_csv(StringIO(csv_text))

            # Очищення назв колонок
            df.columns = [c.strip() for c in df.columns]

            # Видалення рядків де year містить сміття
            df = df[pd.to_numeric(df['year'], errors='coerce').notna()]

            # Приведення типів
            df['year'] = df['year'].astype(int)
            df['week'] = df['week'].astype(int)
            df['VHI']  = pd.to_numeric(df['VHI'], errors='coerce')

            # Видалення VHI = -1
            df = df[df['VHI'] != -1]
            df = df.dropna(subset=['VHI'])

            # Індекс та назва області
            province_id = int(os.path.basename(filename).split('_')[2])
            df['region_id']   = province_id
            df['region_name'] = REGION_NAMES.get(province_id, f"Область {province_id}")

            list_of_dfs.append(df)

        except Exception as e:
            import traceback
            print(f"[ERROR] {filename}: {e}")
            traceback.print_exc()

    if not list_of_dfs:
        raise ValueError("Жоден файл не вдалося прочитати")

    df_all = pd.concat(list_of_dfs, axis=0, ignore_index=True)

    # Перейменовуємо year/week -> Year/Week для консистентності
    df_all = df_all.rename(columns={'year': 'Year', 'week': 'Week'})
    df_all = df_all[['region_id', 'region_name', 'Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']]

    return df_all

df_vhi = clean_data()
print(df_vhi.head(10))
print(f"\nРозмір датафрейму: {df_vhi.shape}")

Empty DataFrame
Columns: [region_id, region_name, Year, Week, SMN, SMT, VCI, TCI, VHI]
Index: []

Розмір датафрейму: (0, 9)


In [10]:
def clean_data(save_dir="vhi_data"):
    all_files = glob.glob(os.path.join(save_dir, "*.csv"))

    if not all_files:
        raise FileNotFoundError(f"Файли не знайдено в папці '{save_dir}'")

    list_of_dfs = []

    for filename in all_files:
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                lines = f.readlines()

            clean_lines = []
            for line in lines:
                # Видаляємо HTML теги
                line = line.replace('<tt><pre>', '').replace('<br>', '').replace('</tt></pre>', '').strip()
                # Видаляємо кому в кінці рядка
                line = line.rstrip(',')
                if line:
                    clean_lines.append(line)

            # Рядок 0: мета-інфо -> пропускаємо
            # Рядок 1: заголовок year,week,SMN,...
            # Рядок 2+: дані
            from io import StringIO
            csv_text = "\n".join(clean_lines[1:])  # з рядку 1 (заголовок)
            df = pd.read_csv(StringIO(csv_text))
            df.columns = [c.strip() for c in df.columns]

            print(f"[DEBUG] колонки: {df.columns.tolist()}, рядків: {len(df)}")

            # Видалення сміттєвих рядків
            df = df[pd.to_numeric(df['year'], errors='coerce').notna()]
            df['year'] = df['year'].astype(int)
            df['week'] = df['week'].astype(int)
            df['VHI']  = pd.to_numeric(df['VHI'], errors='coerce')
            df = df[(df['VHI'] != -1) & df['VHI'].notna()]

            province_id = int(os.path.basename(filename).split('_')[2])
            df['region_id']   = province_id
            df['region_name'] = REGION_NAMES.get(province_id, f"Область {province_id}")

            list_of_dfs.append(df)

        except Exception as e:
            import traceback
            print(f"[ERROR] {filename}: {e}")
            traceback.print_exc()

    if not list_of_dfs:
        raise ValueError("Жоден файл не вдалося прочитати")

    df_all = pd.concat(list_of_dfs, axis=0, ignore_index=True)
    df_all = df_all.rename(columns={'year': 'Year', 'week': 'Week'})
    df_all = df_all[['region_id', 'region_name', 'Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']]

    return df_all

df_vhi = clean_data()
print(df_vhi.head(10))
print(f"\nРозмір датафрейму: {df_vhi.shape}")

[DEBUG] колонки: ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'], рядків: 2237
[DEBUG] колонки: ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'], рядків: 2237
[DEBUG] колонки: ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'], рядків: 2237
[DEBUG] колонки: ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'], рядків: 2237
[DEBUG] колонки: ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'], рядків: 2237
[DEBUG] колонки: ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'], рядків: 2237
[DEBUG] колонки: ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'], рядків: 2237
[DEBUG] колонки: ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'], рядків: 2237
[DEBUG] колонки: ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'], рядків: 2237
[DEBUG] колонки: ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'], рядків: 2237
[DEBUG] колонки: ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'], рядків: 2237
[DEBUG] колонки: ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'], рядків: 2237
[DEB

In [11]:
import re
for f in sorted(glob.glob("vhi_data/*.csv")):
    with open(f, 'r', encoding='utf-8') as file:
        first_line = file.readline()
    province_id = int(os.path.basename(f).split('_')[2])
    name = re.search(r'Province=\s*\d+:\s*(.+?),', first_line)
    if name:
        print(f"NOAA ID={province_id}: {name.group(1).strip()}")

NOAA ID=10: Khmel'nyts'kyy
NOAA ID=11: Kiev
NOAA ID=12: Kiev City
NOAA ID=13: Kirovohrad
NOAA ID=14: Luhans'k
NOAA ID=15: L'viv
NOAA ID=16: Mykolayiv
NOAA ID=17: Odessa
NOAA ID=18: Poltava
NOAA ID=19: Rivne
NOAA ID=1: Cherkasy
NOAA ID=20: Sevastopol'
NOAA ID=21: Sumy
NOAA ID=22: Ternopil'
NOAA ID=23: Transcarpathia
NOAA ID=24: Vinnytsya
NOAA ID=25: Volyn
NOAA ID=2: Chernihiv
NOAA ID=3: Chernivtsi
NOAA ID=4: Crimea
NOAA ID=5: Dnipropetrovs'k
NOAA ID=6: Donets'k
NOAA ID=7: Ivano-Frankivs'k
NOAA ID=8: Kharkiv
NOAA ID=9: Kherson


In [12]:
# Реальний маппінг NOAA ID -> українська назва
NOAA_TO_UA_NAME = {
    1:  "Черкаська",
    2:  "Чернігівська",
    3:  "Чернівецька",
    4:  "Кримська",
    5:  "Дніпропетровська",
    6:  "Донецька",
    7:  "Івано-Франківська",
    8:  "Харківська",
    9:  "Херсонська",
    10: "Хмельницька",
    11: "Київська",
    12: "Київська міська",
    13: "Кіровоградська",
    14: "Луганська",
    15: "Львівська",
    16: "Миколаївська",
    17: "Одеська",
    18: "Полтавська",
    19: "Рівненська",
    20: "Севастополь",
    21: "Сумська",
    22: "Тернопільська",
    23: "Закарпатська",
    24: "Вінницька",
    25: "Волинська",
    # 26: Запорізька - відсутня в NOAA
    # 27: Житомирська - відсутня в NOAA
}

# Новий індекс за українською абеткою (сортуємо назви)
sorted_names = sorted(NOAA_TO_UA_NAME.values())
UA_NAME_TO_INDEX = {name: idx+1 for idx, name in enumerate(sorted_names)}

print("Новий UA індекс -> Назва:")
for name, idx in sorted(UA_NAME_TO_INDEX.items(), key=lambda x: x[1]):
    print(f"  {idx}: {name}")

Новий UA індекс -> Назва:
  1: Івано-Франківська
  2: Волинська
  3: Вінницька
  4: Дніпропетровська
  5: Донецька
  6: Закарпатська
  7: Київська
  8: Київська міська
  9: Кримська
  10: Кіровоградська
  11: Луганська
  12: Львівська
  13: Миколаївська
  14: Одеська
  15: Полтавська
  16: Рівненська
  17: Севастополь
  18: Сумська
  19: Тернопільська
  20: Харківська
  21: Херсонська
  22: Хмельницька
  23: Черкаська
  24: Чернівецька
  25: Чернігівська


In [13]:
def clean_data(save_dir="vhi_data"):
    all_files = glob.glob(os.path.join(save_dir, "*.csv"))
    if not all_files:
        raise FileNotFoundError(f"Файли не знайдено в папці '{save_dir}'")

    list_of_dfs = []

    for filename in all_files:
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                lines = f.readlines()

            clean_lines = []
            for line in lines:
                line = line.replace('<tt><pre>', '').replace('<br>', '').strip()
                line = line.rstrip(',')
                if line:
                    clean_lines.append(line)

            from io import StringIO
            csv_text = "\n".join(clean_lines[1:])
            df = pd.read_csv(StringIO(csv_text))
            df.columns = [c.strip() for c in df.columns]

            df = df[pd.to_numeric(df['year'], errors='coerce').notna()]
            df['year'] = df['year'].astype(int)
            df['week'] = df['week'].astype(int)
            df['VHI']  = pd.to_numeric(df['VHI'], errors='coerce')
            df = df[(df['VHI'] != -1) & df['VHI'].notna()]

            # NOAA ID з імені файлу
            noaa_id = int(os.path.basename(filename).split('_')[2])

            # Українська назва та новий індекс за укр. абеткою
            ua_name  = NOAA_TO_UA_NAME.get(noaa_id, f"Область {noaa_id}")
            ua_index = UA_NAME_TO_INDEX.get(ua_name, noaa_id)

            df['region_id']   = ua_index   # новий індекс за укр. абеткою
            df['region_name'] = ua_name

            list_of_dfs.append(df)

        except Exception as e:
            print(f"[ERROR] {filename}: {e}")

    df_all = pd.concat(list_of_dfs, axis=0, ignore_index=True)
    df_all = df_all.rename(columns={'year': 'Year', 'week': 'Week'})
    df_all = df_all[['region_id', 'region_name', 'Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']]
    df_all = df_all.sort_values(['region_id', 'Year', 'Week']).reset_index(drop=True)

    return df_all

df_vhi = clean_data()
print(df_vhi.head(10))
print(f"\nРозмір: {df_vhi.shape}")
print(f"\nУнікальні області:")
print(df_vhi[['region_id', 'region_name']].drop_duplicates().sort_values('region_id').to_string(index=False))

   region_id        region_name  Year  Week    SMN     SMT    VCI    TCI  \
0          1  Івано-Франківська  1982     1  0.070  255.79  27.63  78.38   
1          1  Івано-Франківська  1982     2  0.073  257.68  30.82  71.96   
2          1  Івано-Франківська  1982     3  0.077  259.75  33.88  64.97   
3          1  Івано-Франківська  1982     4  0.080  261.90  34.23  58.57   
4          1  Івано-Франківська  1982     5  0.082  263.84  32.05  53.87   
5          1  Івано-Франківська  1982     6  0.083  265.41  29.69  51.25   
6          1  Івано-Франківська  1982     7  0.089  267.10  30.04  48.41   
7          1  Івано-Франківська  1982     8  0.095  268.85  31.34  47.04   
8          1  Івано-Франківська  1982     9  0.101  270.94  31.32  46.34   
9          1  Івано-Франківська  1982    10  0.108  273.46  31.52  44.29   

     VHI  
0  53.00  
1  51.39  
2  49.42  
3  46.40  
4  42.96  
5  40.47  
6  39.22  
7  39.19  
8  38.83  
9  37.91  

Розмір: (54650, 9)

Унікальні області:
 r

In [17]:

# ВИБІРКА 1. Ряд VHI для області за вказаний рік

def vhi_by_region_year(df, region_id, year):
    result = df[(df['region_id'] == region_id) & (df['Year'] == year)][['Week', 'VHI']]
    return result

print("=== VHI для Одеської обл. (id=14), 2020 ===")
print(vhi_by_region_year(df_vhi, region_id=14, year=2020))

=== VHI для Одеської обл. (id=14), 2020 ===
       Week    VHI
30344     1  39.27
30345     2  40.51
30346     3  40.52
30347     4  41.39
30348     5  41.29
30349     6  41.17
30350     7  40.68
30351     8  41.47
30352     9  42.42
30353    10  42.58
30354    11  41.62
30355    12  41.38
30356    13  40.63
30357    14  39.13
30358    15  37.68
30359    16  36.42
30360    17  35.46
30361    18  34.79
30362    19  36.20
30363    20  38.63
30364    21  40.35
30365    22  41.58
30366    23  43.71
30367    24  44.47
30368    25  45.25
30369    26  44.58
30370    27  44.39
30371    28  44.71
30372    29  42.95
30373    30  38.89
30374    31  34.91
30375    32  31.49
30376    33  28.25
30377    34  25.75
30378    35  23.88
30379    36  23.03
30380    37  22.77
30381    38  23.57
30382    39  24.60
30383    40  25.93
30384    41  28.94
30385    42  30.52
30386    43  30.62
30387    44  32.91
30388    45  37.40
30389    46  39.60
30390    47  41.77
30391    48  44.29
30392    49  45.80
30393 

In [15]:
# ============================================================
# ВИБІРКА 2: Ряд VHI за діапазон років для вказаних областей
# ============================================================
def vhi_by_region_year_range(df, region_ids, year_start, year_end):
    result = df[
        (df['region_id'].isin(region_ids)) &
        (df['Year'] >= year_start) &
        (df['Year'] <= year_end)
    ][['region_name', 'Year', 'Week', 'VHI']]
    return result

print("=== VHI для Київської та Львівської обл., 2015-2020 ===")
print(vhi_by_region_year_range(df_vhi, region_ids=[9, 13], year_start=2015, year_end=2020))

=== VHI для Київської та Львівської обл., 2015-2020 ===
        region_name  Year  Week    VHI
19154      Кримська  2015     1  40.84
19155      Кримська  2015     2  39.81
19156      Кримська  2015     3  38.63
19157      Кримська  2015     4  37.22
19158      Кримська  2015     5  36.44
...             ...   ...   ...    ...
28205  Миколаївська  2020    48  41.99
28206  Миколаївська  2020    49  43.91
28207  Миколаївська  2020    50  43.54
28208  Миколаївська  2020    51  44.45
28209  Миколаївська  2020    52  45.92

[624 rows x 4 columns]


In [16]:
# ============================================================
# ВИБІРКА 3: Екстремуми, середнє, медіана
# ============================================================
def vhi_extremes(df, region_ids, year_start, year_end):
    subset = df[
        (df['region_id'].isin(region_ids)) &
        (df['Year'] >= year_start) &
        (df['Year'] <= year_end)
    ]
    result = subset.groupby('region_name')['VHI'].agg(
        мін='min',
        макс='max',
        середнє='mean',
        медіана='median'
    ).round(2)
    return result

print("=== Екстремуми VHI для кількох областей, 2010-2024 ===")
print(vhi_extremes(df_vhi, region_ids=[9, 13, 14], year_start=2010, year_end=2024))

=== Екстремуми VHI для кількох областей, 2010-2024 ===
                мін   макс  середнє  медіана
region_name                                 
Кримська      13.54  67.79    40.90    40.79
Миколаївська  19.28  83.00    43.54    42.46
Одеська       18.41  87.16    46.12    45.18
